<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/>
    </div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>How to Run AI-Powered Computer-Aided Engineering Simulations</b></font></h1>
<h2><b>Notebook 2:</b> AI-Powered Crash Simulation Surrogate using PhysicsNeMo</h2>
<br>

## Problem Overview

Automotive crashworthiness assessment is critical in vehicle design. Traditional finite element (FE) simulations (e.g., LS-DYNA) are accurate but computationally expensive, limiting design iteration speed.

**Machine Learning surrogates provide**:
-  **Rapid prediction**: Seconds vs. hours for FE simulations
-  **Scalability**: Test thousands of design variants
-  **Flexibility**: Experiment with different architectures

This notebook demonstrates a **unified pipeline** for crash dynamics modeling using **GeoTransolver**, a geometry-aware transformer that captures spatial relationships and temporal evolution.

###  Learning Objectives

By the end of this notebook, you will understand:
1. How to load and preprocess crash simulation data (Zarr format)
2. GeoTransolver architecture and why it works for physics
3. Autoregressive rollout training for temporal dynamics
4. Evaluation metrics and visualization techniques
5. How to deploy the model for inference

**Framework**: NVIDIA PhysicsNeMo

## 1. Environment Setup

We set up the environment, import necessary libraries, and configure the path to ensure local modules are found.

In [ ]:
# Standard library
import os
import sys
import time
import json
import warnings
from pathlib import Path

# Third-party libraries
import numpy as np
import torch
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import zarr

# Hydra configuration
from hydra.utils import instantiate
from omegaconf import DictConfig, OmegaConf, open_dict

# PhysicsNeMo utilities
from physicsnemo.utils.checkpoint import load_checkpoint, save_checkpoint

# Visualization libraries
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Local utils modules
utils_path = os.path.abspath("/workspace/utils")
if utils_path not in sys.path:
    sys.path.append(utils_path)
    
import rollout 
import datapipe
from datapipe import CrashPointCloudDataset, simsample_collate
from zarr_reader import Reader
from plotting_utils import plot_l2_errors, save_trajectory_to_vtp
from gale_demonstration import (
    load_ahmed_body_geometry, run_mlp_drift_comparison , plot_mlp_drift_comparison
)

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

## Transolver & GeoTransolver: Efficient Physics Transformers

### The Transformer Challenge for Physics

Standard transformers revolutionized NLP, vision, and audio, but face critical challenges for high-resolution physics:

**Computational Complexity**: O(N²) for N nodes/points
- Crash simulations: ~13,000 nodes
- Standard attention: 169 million pairwise comparisons per layer!
- GPUs: Run out of memory or become prohibitively slow

### Transolver's Solution: Physics-Aware Slicing

**Transolver** introduces **physics-based spatial partitioning** to reduce complexity:

**Key Ideas**:
1. **Slice-based attention**: Partition space into meaningful physics regions (slices)
2. **Local + Global**: Capture both fine details and large-scale interactions
3. **Complexity**: O(N log N) instead of O(N²)

**Three Components**:
- **Physics-aware slicing**: Divide space based on physical structure
- **Global context integration**: Design parameters (velocity, material) influence all nodes
- **Rollout**: Predict dynamics over time with temporal consistency

### GeoTransolver: Adding GALE to Transolver

**GeoTransolver** extends Transolver by addressing a second critical problem: **representation drift**.

#### The Representation Drift Problem

In standard deep networks, each layer only sees the output of the previous layer:
```
Layer 1 → Hidden State → Layer 2 → Hidden State → ... → Output
```
As information flows through 6+ layers, **geometric structure gets lost or distorted**.

#### GALE: Geometry-Aware Latent Embeddings

**Solution**: Re-inject original geometry at every layer to maintain spatial grounding:

```
┌─────────────┐
│  Geometry   │────┬─> Layer 1 ─┬─> Layer 2 ─┬─> ... ─> Output
│ (positions) │    │            │            │
└─────────────┘    │            │            │
   Persistent ─────┴────────────┴────────────┘
   Grounding
```

**Why This Matters**:
- **Preserves geometry**: Every layer has direct access to spatial structure
- **Prevents drift**: Car shape doesn't dissolve into random point cloud
- **Physics consistency**: Maintains structural relationships through deep network


### GeoTransolver Architecture Flow

```
Input: Coordinates (N×3) + Global Features (velocity, thickness, etc.)
   ↓
Geometry Encoding (persistent reference)
   ↓
┌─────────────────────────────────┐
│ Transformer Stack (6 layers)    │
│  Each layer receives:           │
│  • Previous hidden state        │
│  • Original geometry (GALE)     │  ← Key difference from Transolver!
│  • Global context               │
│  • Physics-aware slicing        │
└─────────────────────────────────┘
   ↓
Predicted Accelerations (N×3) + Physical Fields (Nx2)
   ↓
Rollout Integration (50 timesteps)
```

**Why GeoTransolver for Crash Simulation**: 
- **Transolver efficiency**: Handles 13,000 nodes efficiently
- **GALE robustness**: Maintains structural integrity through deep network
- **Physics consistency**: Captures both local deformation and global motion

**For deeper understanding**: See [Transolver Tutorial: From Theory to Training](https://github.com/openhackathons-org/End-to-End-AI-for-Science/tree/transolver-ag/workspace/python/jupyter_notebook/Transolver) for a full set of notebooks explaining the Transolver and GeoTransolver architectures.

To build intuition for **representation drift**, we construct a **simplified but realistic simulation** using actual neural network components (MLP layers) applied to the *Ahmed Body* geometry. The goal is **conceptual clarity**: to visually illustrate how geometric representations can progressively degrade—or remain stable—across network depth.

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Experiment | Method | Expected Outcome |
|------------|--------|------------------|
| **Case A** | Standard Network (No GALE) | Geometry becomes progressively **noisy/fuzzy** as depth increases |
| **Case B** | GALE Network (Context Injection) | Geometry remains **clean and sharp** throughout all layers |

</div></div>

**Key Point:** Both networks receive **identical noise and MLP transformations**. The *only* difference is GALE's persistent context injection at each layer.

To quantify drift, we compute:
- **Mean Squared Error (MSE):** Distance from original geometry
- **Noise Level (Std Dev):** Spread of point displacements

This demonstration uses **actual neural network components**:

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Component | Purpose |
|-----------|---------|
| **MLP Layers** | Simulates learned transformations in real networks |
| **Random Noise** | Represents accumulated perturbations from optimization |
| **Context Injection** | Simplified GALE mechanism |

</div></div>

#### Parameters:

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `num_layers` | 30 | Number of network layers to simulate |
| `hidden_dim` | 64 | MLP hidden layer dimension |
| `noise_scale` | 0.04 | Per-layer noise standard deviation |
| `context_strength` | 0.6 | GALE's pull toward original  |

</div></div>


In [ ]:
# Load simple geometry
geometry = load_ahmed_body_geometry("/workspace/data/ahmed_body", 40)

# Parameters for visible noise accumulation
results = run_mlp_drift_comparison(
    geometry,
    num_layers=30,
    hidden_dim=64,
    noise_scale=0.02,      # Higher = more visible noise
    context_strength=0.6   # GALE correction strength
)

plot_mlp_drift_comparison(results, geometry, [0, 10, 20, 30])

## 2. Configuration

### Hierarchical Configuration System

We use **Hydra/OmegaConf** for managing hyperparameters with:
-  Type safety and validation
-  Easy experiment tracking
-  Reproducible results

### 2. Load Configuration

Now that you have processed the crash simulation dataset into zarr files in the first notebook, we must create a configuration that defines how the model will be trained.

Ordinarily, training is managed via Hydra configurations located in `conf/`, but in this notebook we set all the parameters directly in a Python dictionary for transparency and ease of modification.

Since we cannot pass command-line arguments in a notebook, we use `OmegaConf.create()` to load the configuration programmatically. We will build up a dictionary that contains all our training parameters, model architecture settings, and data paths in one place. This is to makes the configuration transparent and easy to modify directly in the notebook, rather than managing multiple YAML files across different directories.

Here is a breakdown of what each section does:

#### Configuration Structure

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Section | Purpose | Key Parameters |
|---------|---------|----------------|
| **reader** | Data loading | Zarr format, file paths |
| **datapipe** | Preprocessing | Normalization, features, batch size |
| **model** | Architecture | Layers, attention, dimensions |
| **training** | Optimization | Learning rate, epochs, validation |

</div></div>

### 2.1 Data Configuration

#### Zarr Data Format

**Zarr** is a cloud-optimized array storage format ideal for scientific data:
-  **Chunked storage**: Efficient partial reads
-  **Parallel I/O**: Multi-threaded loading
-  **Compression**: Reduce storage by 10-100×

#### Point Cloud Dataset

**Why point clouds?** Crash simulations have irregular meshes with varying connectivity. Point cloud representation is:
- **Flexible**: No fixed mesh topology required
- **Efficient**: Only store node positions, not edges
- **Compatible**: Works with transformer attention

**Key Parameters Explained**:

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `num_samples` | 8 | Number of crash simulations (108 available) |
| `num_steps` | 51 | Timesteps per simulation (0-250ms @ 5ms intervals) |
| `features` | `[]` | No additional node features (coordinates only) |
| `global_features` | 3 features | Design parameters: |
| | `velocity_x` | Impact velocity (km/h) |
| | `thickness_scale` | Material thickness multiplier |
| | `rwall_origin_y` | Rigid barrier position (mm) |

</div></div>

**Why 20 samples?** Quick training demo. Increase to 108 for full dataset.

**Why 51 timesteps?** Captures 250ms crash event (typical airbag deployment time), intial state + 50 steps.



In [ ]:
DATA_DIR = "/workspace/data/processed_data_local/" #replace with "/workspace/data/processed_data_vtp/" if you completed notebook 1
GLOBAL_FEATURES_FILE = "/workspace/stats/global_features.json"
NUM_SAMPLES = 20
NUM_STEPS = 51
SPATIAL_COORDINATES = 3
PHYSICS_FEATURES = 2
OUTPUT_FEATURES = SPATIAL_COORDINATES + PHYSICS_FEATURES # plastic_strain + von_mises_stress

# Initialize configuration with an empty dictionary
config_dict = {}

# Add DATA PIPELINE
config_dict["datapipe"] = {
    "_target_": "datapipe.CrashPointCloudDataset",
    "data_dir": DATA_DIR,
    "num_samples": NUM_SAMPLES,
    "num_steps": NUM_STEPS,
    "features": ["plastic_strain", "von_mises_stress"], # as well as spatial coordinates
    "global_features_filepath": GLOBAL_FEATURES_FILE,
    "global_features": ["velocity_x", "thickness_scale", "rwall_origin_y"],
}

### 2.2 Model Configuration

#### GeoTransolver Hyperparameters

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Parameter | Value | What It Controls |
|-----------|-------|------------------|
| `functional_dim` | 3 | Input features per node (x, y, z) |
| `geometry_dim` | 3 | Spatial embedding size |
| `global_dim` | 3 | Number of global context features |
| `out_dim` | 250 | Output size (50 timesteps × (3 coords + 2 fields)) |
| `slice_num` | 128 | Physics-aware attention partitions |
| `n_layers` | 6 | Transformer depth |
| `n_hidden` | 512 | Hidden layer width |
| `use_te` | False | Transformer Engine (disable for compatibility) |
| `dt` | 5e-3 | Timestep size (5 milliseconds) |

</div></div>

#### Understanding Key Parameters

**`slice_num=128`**: How many spatial regions for attention?
- Too few: Miss local details
- Too many: Computational overhead
- 128: Good balance for ~13k nodes

**`n_layers=6`**: Transformer depth
- Deeper: More expressive, captures complex patterns
- Shallower: Faster training, less overfitting risk
- 6: Standard for physics applications

**`dt=5e-3`**: Timestep integration
- Smaller: More accurate integration, slower
- Larger: Faster but unstable
- 5ms: Matches FE simulation timestep


In [ ]:
# Architecture parameters for GeoTransolver
config_dict["model"] = {
    "_target_": "rollout.GeoTransolverRolloutTraining",
    "_convert_": "all",
    "functional_dim": OUTPUT_FEATURES,
    "out_dim": (NUM_STEPS - 1) * OUTPUT_FEATURES,
    "geometry_dim": SPATIAL_COORDINATES,
    "global_dim": SPATIAL_COORDINATES,
    "slice_num": 128,
    "n_layers": 6,
    "use_te": False,
    "num_time_steps": NUM_STEPS,
    "dt": 5e-3,
}


### 2.3 Training Configuration

#### Optimization Strategy

**Adam Optimizer** with **Cosine Annealing LR**:
- Start: High learning rate (1e-4) for fast initial learning
- End: Low learning rate (1e-6) for fine-tuning
- Schedule: Smooth cosine decay prevents sudden jumps

**Why Cosine Annealing?**
- Smoother than step decay
- Helps escape local minima
- Better final convergence

#### Training Parameters

<div style="display: flex; flex-direction: row;"><div style="float: left;">

| Setting | Value | Purpose |
|---------|-------|---------|
| `epochs` | 10 | Number of full passes through data |
| `start_lr` | 1e-4 | Initial learning rate |
| `end_lr` | 1e-6 | Final learning rate |
| `amp` | True | Automatic Mixed Precision (2× speedup) |
| `save_freq` | 1 | Save checkpoint every epoch |
| `val_freq` | 1 | Validate every epoch |

</div></div>

**Performance Tip**: AMP uses FP16 for speed, FP32 for stability automatically.


In [ ]:
EPOCHS = 3 # Small number for training (in this notebook)
VALIDATION_FREQ = 1 # Run validation every epoch (for testing)
SAVE_CHECKPOINT_FREQ = 1 # Save checkpoint every epoch (for testing)
CHECKPOINT_PATH = "/workspace/outputs/checkpoints"

# TRAINING CONFIGURATION
config_dict["training"] = {
    # Training parameters
    "num_time_steps": config_dict["datapipe"]["num_steps"],
    "num_training_samples": config_dict["datapipe"]["num_samples"],
    "num_validation_samples": config_dict["datapipe"]["num_samples"],
    # Learning rate schedule
    "start_lr": 0.0001,
    "end_lr": 0.0000003,
    # Training schedule
    "epochs": EPOCHS, 
    "validation_freq": VALIDATION_FREQ,
    "save_checkpoint_freq": SAVE_CHECKPOINT_FREQ,
    # Performance optimization
    "amp": True,
    "use_apex": True,
    "num_dataloader_workers": 4,
    # Logging & checkpointing
    "ckpt_path": CHECKPOINT_PATH,
}

cfg = OmegaConf.create(config_dict)
print("Configuration loaded successfully.")

# Access values like:
print(f'epochs: {cfg.training.epochs}')
print(f'n_layers: {cfg.model.n_layers}')

## 3. Training Functions

### Training Pipeline Overview

In PhysicsNeMo the trainer is abstracted away from you, so all you have to do it update the config file.

The trainer class looks like this:

```python
class Trainer
"""Trainer for crash simulation models with unified SimSample input."""
def __init__(self, cfg: DictConfig, logger0: RankZeroLoggingWrapper):
    assert DistributedManager.is_initialized()
    self.dist = DistributedManager()
    self.cfg = cfg
    self.rollout_steps = cfg.training.num_time_steps - 1
    self.amp = cfg.training.amp 

    model_name = cfg.model._target_
    datapipe_name = cfg.datapipe._target_
```

And is called like this:

```python
trainer = Trainer(cfg, logger0)
```

In this notebook we are going to define all the functions within `Trainer` to get you familar with how they work! These define the:

1. Batch Loading                    

2. Forward Pass                     
    • Model predicts accelerations   
    • Integrate to full trajectory   

3. Compute Loss                     
    • MSE between pred & ground truth

4. Backward Pass                    
    • Compute gradients              
    • Update weights                 


### Key Functions

1. **`initialize_data`**: Load dataset, compute normalization stats
2. **`train_step`**: One training iteration (forward + backward)
3. **`forward`**: Model prediction and loss computation
4. **`backward`**: Gradient computation and weight update
5. **`validate`**: Evaluate on validation set (no gradient)


#### Training Step Components

**Forward Pass**:
- Input: Node coordinates + global features
- Model: Predicts accelerations
- Integration: Convert accelerations → velocities → positions
- Output: Full trajectory (50 timesteps)

**Loss Computation**:
- Compare predicted positions vs. ground truth
- MSE (Mean Squared Error) across all nodes and timesteps
- Backpropagate through entire rollout sequence

**Backward Pass**:
- Automatic differentiation via PyTorch
- Gradient clipping (if enabled) prevents exploding gradients
- AMP: Mixed precision for speed


In [ ]:
def train_step(sample, model, optimizer, scaler, criterion, data_stats, cfg):
    # Return to training mode
    model.train()
    amp = cfg.training.amp
    rollout_steps = cfg.training.num_time_steps - 1
    
    optimizer.zero_grad()

    loss = forward(sample, model, criterion, data_stats, rollout_steps, amp)

    backward(loss, optimizer, scaler, amp)
    return loss.item() 

def forward(sample, model, criterion, data_stats, rollout_steps, amp):
    with torch.amp.autocast(device_type="cuda", enabled=amp):
        # Forward pass
        pred = model(sample=sample, data_stats=data_stats)
        
        # Reshape target (Target is already normalized by the datapipe)
        target_flat = sample.node_target 
        N, Fo = target_flat.size(0), 5
        target = target_flat.view(N, rollout_steps, Fo).transpose(0, 1).contiguous() 

        loss = criterion(pred, target)
    
        return loss

def backward(loss, optimizer, scaler, amp):
    if amp:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

#### Validation Function

**Purpose**: Check generalization to unseen data

**Key Differences from Training**:
-  No gradient computation (`@torch.no_grad()`)
-  No backpropagation
-  Model in eval mode (disables dropout)
-  Full dataset evaluation

**Metrics Computed**:
- MSE: Mean Squared Error across all timesteps
- Per-timestep errors: Track how error accumulates over time


In [ ]:
@torch.no_grad()
def validate(model, dataloader, criterion, data_stats, cfg, device):

    model.eval()
    total_mse = 0.0
    count = 0
    
    amp = cfg.training.amp
    rollout_steps = cfg.training.num_time_steps - 1

    # Loop through the validation dataloader
    for batch in dataloader:
        # Extract sample and move to device
        sample = batch[0].to(device)
        
        # Use the same 'forward' helper used in training for consistency
        loss = forward(sample, model, criterion, data_stats, rollout_steps, amp)
        
        total_mse += loss.item()
        count += 1

    # Calculate mean across all validation samples
    avg_mse = total_mse / count if count > 0 else 0.0
        
    return {"MSE": avg_mse}

## 4. Load Training Data

Initialize the data pipeline and examine a sample.

**What Happens Here**:
1. **Reader**: Loads Zarr files from disk
2. **Dataset**: Processes raw data, computes normalization statistics
3. **DataLoader**: Creates batches for training
4. **Stats**: Move normalization parameters to GPU

**Normalization**: Crucial for training stability. We normalize:
- Positions: Mean-center and scale by std dev
- Velocities: Normalize derivatives
- Features: Per-feature normalization


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def stats_to_device(obj, device):
    """Function to move normalization stats dictionary to GPU"""
    if not obj:
        return {}
    return {
        k: (v.to(device) if isinstance(v, torch.Tensor) else torch.as_tensor(v, dtype=torch.float32, device=device))
        for k, v in obj.items()
    }

z_files = sorted([f for f in os.listdir(cfg.datapipe.data_dir) if f.endswith('.zarr')])
z_meta = zarr.open(os.path.join(cfg.datapipe.data_dir, z_files[0]), mode='r')
at = z_meta.attrs

physics_stats = {
    "plastic_strain_max": float(at["plastic_strain_max"]),
    "von_mises_stress_max": float(at["von_mises_stress_max"]),
}

full_dataset = CrashPointCloudDataset(
    data_dir=cfg.datapipe.data_dir,
    reader=Reader(),
    num_samples=len(z_files),
    num_steps=cfg.datapipe.num_steps,
    features=cfg.datapipe.features,
    global_features_filepath=cfg.datapipe.global_features_filepath,
    global_features=cfg.datapipe.global_features,
    physics_stats=physics_stats,
)

data_stats = {
    "node": stats_to_device(full_dataset.node_stats, device),
    "feature": {
        k: {
            stat: torch.tensor(at[f"{k}_{stat}"], device=device, dtype=torch.float32)
            for stat in ["min", "max"]
        }
        for k in ["plastic_strain", "von_mises_stress"]
    },
}

Now lets build our datasets:

In [ ]:
VAL_RUNS = [1, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130]

val_indices = [i for i in VAL_RUNS if i < len(z_files)]
train_indices = [i for i in range(len(z_files)) if i not in val_indices]

train_sub = torch.utils.data.Subset(full_dataset, train_indices)
val_sub = torch.utils.data.Subset(full_dataset, val_indices)

train_loader = torch.utils.data.DataLoader(
    train_sub,
    batch_size=1,
    shuffle=True,
    num_workers=cfg.training.num_dataloader_workers,
    collate_fn=simsample_collate,
)
val_loader = torch.utils.data.DataLoader(
    val_sub,
    batch_size=1,
    shuffle=False,
    collate_fn=simsample_collate,
)

train_dataloader, val_dataloader = train_loader, val_loader
print(f"train samples: {len(train_indices)}, val samples: {len(val_indices)}")

## 5. Train the Model

Initialize model, optimizer, and scheduler.

In [ ]:
model = instantiate(cfg.model).to(device)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.training.start_lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.training.epochs, eta_min=cfg.training.end_lr
)

scaler = GradScaler(enabled=cfg.training.amp)

Lets see how many parameters are in the model:

In [ ]:
def count_parameters(model: torch.nn.Module):
    return sum(p.numel() for p in model.parameters())

n_params = count_parameters(model)
print(f"Number of trainable parameters: {n_params:,}")

### 5.1 Training Loop

**What to Watch**:
1. **Loss**: Should decrease steadily (not monotonically)
2. **Learning Rate**: Decays following cosine schedule
3. **Validation Error**: Should track training loss
4. **Time per Epoch**: Consistent timing indicates stable training

**Common Issues**:
- Loss → NaN: Learning rate too high or data normalization issue
- Loss plateaus early: Increase model capacity or learning rate
- Validation >Training: Overfitting (need more data or regularization)

### Checkpoint Management

`physicsnemo` provides useful checkpointing utilities.

**What's Saved**:
- Model weights
- Optimizer state (momentum, adaptive learning rates)
- Scheduler state (learning rate schedule)
- Scaler state (AMP loss scaling)
- Epoch number


In [ ]:
STATE_FILE = "training_state.json"
SAVE_FREQ = cfg.training.save_checkpoint_freq
VAL_FREQ = cfg.training.validation_freq
TOTAL_EPOCHS = cfg.training.epochs

# --- 1. Smart Resumption ---
if os.path.exists(STATE_FILE):
    print(f"🔄 Resuming from {STATE_FILE}...")
    with open(STATE_FILE, 'r') as f:
        state = json.load(f)
else:
    print("✨ Starting fresh run...")
    state = {
        "epoch": 0,
        "train_mse_history": [],  # Renamed for clarity
        "val_mse_history": [],
        "best_val_mse": float('inf')
    }

start_epoch = state["epoch"] + 1

# --- 2. Training Loop ---
print(f"Training: Epoch {start_epoch} -> {TOTAL_EPOCHS}")

try:
    pbar = tqdm(range(start_epoch, TOTAL_EPOCHS + 1), initial=start_epoch, total=TOTAL_EPOCHS, unit="ep")
    
    for epoch in pbar:
        model.train()
        epoch_loss = 0.0
        num_batches = 0
        
        # --- Batch Training ---
        for sample in train_dataloader:
            sample = sample[0].to(device) if isinstance(sample, (list, tuple)) else sample.to(device)
            
            # Assuming train_step returns scalar loss (MSE)
            loss = train_step(sample, model, optimizer, scaler, criterion, data_stats, cfg)
            
            # .item() prevents GPU memory leaks
            epoch_loss += loss.item() if isinstance(loss, torch.Tensor) else loss
            num_batches += 1

        scheduler.step()
        
        # --- Metrics Calculation ---
        current_train_mse = epoch_loss / max(num_batches, 1)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update State
        state["epoch"] = epoch
        state["train_mse_history"].append(current_train_mse)

        # --- Validation ---
        val_mse = None
        if epoch % VAL_FREQ == 0:
            val_stats = validate(model, val_dataloader, criterion, data_stats, cfg, device)
            val_mse = val_stats['MSE']
            
            state["val_mse_history"].append({"epoch": epoch, "mse": val_mse})
            
            # Save "Best" Model
            if val_mse < state["best_val_mse"]:
                state["best_val_mse"] = val_mse
                # Optional: Save specific 'best' checkpoint here if desired
                # save_checkpoint(..., suffix="_best")

        # --- Periodic Checkpoint ---
        if epoch % SAVE_FREQ == 0:
            save_checkpoint(cfg.training.ckpt_path, models=model, optimizer=optimizer, scheduler=scheduler, scaler=scaler, epoch=epoch)

        # --- Sync JSON (Fail-safe) ---
        with open(STATE_FILE, 'w') as f:
            json.dump(state, f, indent=4)

        # --- Update Progress Bar ---
        # Shows real-time Train MSE, Val MSE (if avail), and LR
        metrics = {
            "Tr_MSE": f"{current_train_mse:.2e}",
            "LR": f"{current_lr:.2e}"
        }
        if val_mse:
            metrics["Val_MSE"] = f"{val_mse:.2e}"
            
        pbar.set_postfix(metrics)

except KeyboardInterrupt:
    print("\nPaused by user.")
except Exception as e:
    print(f"\nCRASH: {e}")
    raise e
finally:
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=4)
    print("Progress saved.")

## 6. Inference and Evaluation

Load a trained checkpoint and evaluate on test data.

### Checkpoint Loading

Excellent! Our training and validation losses are going down with training time. Rather than train a model to convergence here we will load a pre-trained checkpointand run inference on test data to evaluate model performance. This model was trained for 6000 epochs. First we must load the checkpoint, `physicsnemo` provides a utility for loading checkpoints. The checkpoint contains all necessary state to reproduce the model's predictions.


In [ ]:
checkpoint_path = cfg.training.ckpt_path

load_checkpoint(
    checkpoint_path,
    models=model,
    device=device,
)

# Set model to evaluation mode
model.eval()
print(f"Loaded checkpoint from {checkpoint_path}")

### Test Dataset

Create a separate test dataset to evaluate generalization:
- Different crash scenarios than training
- No gradient computation
- Model in evaluation mode


### Run Inference

Generate predictions and compute error metrics.

**Denormalization**: Convert normalized predictions back to physical units (millimeters, seconds).

**Error Metrics**:
- **L2 Error**: Euclidean distance between predicted and actual positions
- **Relative Error**: L2 error normalized by displacement magnitude
- **Per-timestep**: Track how error accumulates over time

In [ ]:
def denormalize(y, data_stats):
    """
    Denormalizes XYZ (Mean-Std) and Physics (Global Max).
    """
    if y.ndim == 2:
        view_shape = (1, -1)
    else:
        view_shape = (1, 1, -1)

    # 1. Denormalize Geometry (Standardization)
    pos_mean = data_stats['node']['pos_mean'].view(*view_shape)
    pos_std = data_stats['node']['pos_std'].view(*view_shape)
    coords = y[..., :3] * pos_std + pos_mean
    
    # 2. Denormalize Physics (Global Max Scaling)
    strain_max = data_stats['feature']['plastic_strain']['max'].view(*view_shape)
    stress_max = data_stats['feature']['von_mises_stress']['max'].view(*view_shape)
    
    strain = y[..., 3:4] * strain_max
    stress = y[..., 4:5] * stress_max
    
    return torch.cat([coords, strain, stress], dim=-1)

predictions = []
ground_truths = []

with torch.no_grad():
    for idx, batch in enumerate(val_dataloader):
        sample = batch[0].to(device)
        
        # Forward pass (match train_ddp: autocast when amp enabled)
        with torch.amp.autocast(device_type="cuda", enabled=cfg.training.amp):
            pred_seq = model(sample=sample, data_stats=data_stats)  # [T, N, 5]
        
        # Ground truth
        N = sample.node_target.size(0)
        T = cfg.training.num_time_steps - 1
        gt_seq = sample.node_target.view(N, T, 5).transpose(0, 1)  # [T, N, 5]
        
        # Denormalize
        pred_denorm = denormalize(pred_seq, data_stats)
        gt_denorm = denormalize(gt_seq, data_stats)
        
        # Compute L2 error per timestep
        l2_error = torch.norm(pred_denorm - gt_denorm, dim=-1).mean(dim=1)  # [T]
        gt_norm = torch.norm(gt_denorm, dim=-1).mean(dim=1)  # [T]
        
        predictions.append(pred_denorm.cpu())
        ground_truths.append(gt_denorm.cpu())
        
        if (idx + 1) % 5 == 0:
            print(f"Processed {idx + 1}/{len(val_dataloader)} samples")

print(f"\nInference complete!")

In [ ]:
# Compute and Plot L2 Error Over Time
error_stats = plot_l2_errors(predictions, ground_truths)

## Storing Predictions

We can export predictions and ground truth to VTP files for 3D visualization in ParaView or KitCAE.

In [ ]:
OUTPUT_DIR_PRED = '/workspace/outputs/vtps/predicted_vtps'
OUTPUT_DIR_GT =  '/workspace/outputs/vtps/ground_truth_vtps'

# Update your save loop
for i in range(min(3, len(predictions))):
    save_run_as_single_vtp(predictions[i], OUTPUT_DIR_PRED, i+1)
    save_run_as_single_vtp(ground_truths[i], OUTPUT_DIR_GT, i+1)
    print(f"Saved run {i+1}")

We can generate Animation GIFs, Side-by-side comparison of ground truth vs prediction for displacement, plastic strain and von_mises_stress fields.

This cell runs a script to avoid issues with notebook memory allocation.

In [ ]:
OUTPUT_DIR_PRED = '/workspace/outputs/vtps/predicted_vtps'
OUTPUT_DIR_GT =  '/workspace/outputs/vtps/ground_truth_vtps'
FRAME_DIR = '/workspace/outputs/frames' 

# Ensure the frame directory exists
os.makedirs(FRAME_DIR, exist_ok=True)

# Run as a script (due to issues with notebook memory) with virtual frame buffer wrapper
!xvfb-run -a python /workspace/utils/render_comparison.py \
    --pred_root $OUTPUT_DIR_PRED \
    --gt_root $OUTPUT_DIR_GT \
    --save_path $FRAME_DIR \
    --modes Displacement Plastic_Strain Von_Mises_Stress

Now we can view these predictions and see how well the surrogate captures the dynamics!

In [ ]:
for field in ['displacement', 'plastic_strain', "von_mises_stress"]:
    display(Image(filename=f"{FRAME_DIR}/crash_{field}.gif"))

## 7.  Summary

Congratulations! You've trained an AI-surroage the crash simulation and validated the outputs!

### What You Learned

 **Data Processing**: Zarr format, normalization, point cloud representation  
 **GeoTransolver**: Geometry-aware attention for physics  
 **Rollout Training**: Temporal dynamics with integration  
 **Evaluation**: Error metrics and visualization  
 **Deployment**: Inference and result analysis  

### Next Steps

####  Experiments to Try
1. **Increase dataset size**: Use all 108 samples for better generalization
2. **Tune hyperparameters**: Try different learning rates, layer depths
3. **Ablation studies**: Remove global features, change attention slices
4. **Architecture variants**: Compare with MeshGraphNet or FNO

####  Checkout PhysicsNeMo for:
- **Multi-GPU training**: Scale to larger datasets
- **Transfer learning**: Fine-tune on a pretrained model on a new dataset
- **Uncertainty quantification**: Ensemble predictions to anaylse where the model is more/less confident. 
- **Real-time deployment**: Real-Time Digital Twin using Omniverse Kit-CAE

####  Resources
- [PhysicsNeMo Documentation](https://docs.nvidia.com/physicsnemo)
- [GeoTransolver Paper](https://arxiv.org/abs/...)